In [15]:
import docling
import langchain_core
from typing import Iterator
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document as LCDocument
from langchain_text_splitters import RecursiveCharacterTextSplitter
from docling.document_converter import DocumentConverter
from tempfile import TemporaryDirectory
from langchain_huggingface import HuggingFaceEndpoint
from typing import Iterable
from langchain_core.documents import Document as LCDocument
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain import hub
import pandas as pd
import math
import ast
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['HF_HOME'] = '/blue/rcstudents/smaley/alp/ALP_updated/models' # for huggingface model install
HF_API_KEY = os.environ.get("PERSONAL_HUGGING_FACE_API_KEY")

DATA_DIR = "../data"
reldir = f'{DATA_DIR}/Laws'

In [16]:
class DoclingPDFLoader(BaseLoader):

    def __init__(self, file_path: str | list[str]) -> None:
        self._file_paths = file_path if isinstance(file_path, list) else [file_path]
        self._converter = DocumentConverter()

    def lazy_load(self) -> Iterator[LCDocument]:
        for source in self._file_paths:
            dl_doc = self._converter.convert(source).document
            text = dl_doc.export_to_markdown()
            yield LCDocument(page_content=text)

In [17]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
import os
import re

i = 0

file = "CA SB 100.pdf"

print(f"\n{file}")
print(f"Loading pages from {file}...")
path = reldir + "/" + file
loader = DoclingPDFLoader(file_path=path)

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
docs = loader.load()
splits = text_splitter.split_documents(docs)

if not splits:
    print("ERROR: No splits generated.")
else:
    print(f"Generated {len(splits)} splits.")

# Load embedding model
HF_EMBED_MODEL_ID = "BAAI/bge-large-en-v1.5"
embeddings = HuggingFaceEmbeddings(model_name=HF_EMBED_MODEL_ID)

# Build FAISS vectorstore
reference_law = file[:-4]
FAISS_INDEX_FILE = reference_law

vectorstore = FAISS.from_documents(splits, embeddings)

# Save FAISS index to a file
vectorstore.save_local(FAISS_INDEX_FILE)

print(f"FAISS index saved to {FAISS_INDEX_FILE}")


CA SB 100.pdf
Loading pages from CA SB 100.pdf...
Generated 62 splits.
FAISS index saved to CA SB 100


In [18]:
vector_store = FAISS.load_local(reference_law, embeddings, allow_dangerous_deserialization=True)

In [19]:
DATA_DIR = "../data"
REENCODED_CREDIT_MULTIPLES = pd.read_excel(f'{DATA_DIR}/reencoded_col_BD_from_database.xlsx')
TECHNOLOGY_MAPPINGS = pd.read_excel(f'{DATA_DIR}/technology_mapping.xlsx')

# CLAUDE_API_KEY = os.getenv("CLAUDE_API_KEY")
# LLAMA_API_KEY = os.getenv("LLAMA_API_KEY")
# HF_API_KEY = os.getenv("HUGGING_FACE_API_KEY")

UF_API_KEY = os.getenv("UF_API_KEY")
BASE_URL = "https://api.ai.it.ufl.edu/"

QUESTIONS_DIR = f'{DATA_DIR}/Questions.xlsx'
QUESTIONS_DF = pd.read_excel(QUESTIONS_DIR, sheet_name="Questions")
PROMPT_ENGINEERING = pd.read_excel(QUESTIONS_DIR, sheet_name="Prompt Engineering").columns[0]
FILES_TO_QUERY_TXT = "files_to_query.txt"

GT_DF = pd.read_excel(f'{DATA_DIR}/Database June 20 2022.xlsx', sheet_name='master')

MODELS = [
    # "gemma-1.1-7b-it", # doesnt work
    "llama-3.0=70b-instruct",
    # "llama-3.0-8b-instruct", # doesnt work
    "mixtral-8x7b-instruct",
    "mistral-7b-instruct",
    "gpt-3.5-turbo",
    "gpt-4-turbo",
    "gpt-4o",
    "gpt-4o-mini"
    # "claude-3.5-sonnet" # need to request access for
]

OUTPUT_DIR = "../output/docling_12_12_24"

In [20]:
def retrieve_ground_truth_values(df, id, reference_law):
    row = df.loc[df['reference law'] == reference_law]

    if id == 3: # credit multiplier
        str_multiple_list = REENCODED_CREDIT_MULTIPLES.loc[REENCODED_CREDIT_MULTIPLES['rps_law'] == reference_law]["credit_multiplier"].iloc[0]
        return ast.literal_eval(str_multiple_list)

    elif id == 4: # categories of sales excluded or exempt
    # headings = df.loc[df['reference law'] == 'area names'].iloc[:, 13:19].columns.values # columns N-R
        cats_of_sales = row.iloc[:, 16].values[0] # col Q
        cats_of_sales = cats_of_sales.split(", ")
        # excluded_or_exempt = dict(zip(headings, cats_of_sales))
        return cats_of_sales

    elif id == 5: # date commitment passed into law
        date_commit = str(row.iloc[:, 5].values[0])[:4] # column F
        return date_commit

    elif id == 6: # RPS and ces commitment and year that it must be met
        rps_commit_perc = row.iloc[:, 8:10].values[0]
        for i in range(len(rps_commit_perc)):
            if math.isnan(rps_commit_perc[i]):
                rps_commit_perc[i] = "None"
        rps_val = f"{rps_commit_perc[0]}:{rps_commit_perc[1]}"
        if rps_val == "None:None": # if CES commit is non-existant
            rps_commit_perc = row.iloc[:, 6:8].values[0]
            for i in range(len(rps_commit_perc)):
                if math.isnan(rps_commit_perc[i]):
                    rps_commit_perc[i] = "None"
            rps_val = f"{rps_commit_perc[0]}:{rps_commit_perc[1]}"
        return rps_val

    elif id == 7: # RPS initial commit as a %
        rps_initial = row.iloc[:, 10].values[0]
        return rps_initial

    elif id == 8: # % of RPS and CES that is voluntary, one value for both
        rps_voluntary = row.iloc[:, 17].values[0]
        return rps_voluntary

    elif id == 10: # energy sources
        energy_sources_all = row.iloc[:, 135:172]
        energy_sources = []
        for column in energy_sources_all.columns:
            if pd.notna(energy_sources_all[column].values[0]):
                energy_sources.append(column)
        ret = []
        for energy_source in energy_sources:
            ret.append(TECHNOLOGY_MAPPINGS.loc[TECHNOLOGY_MAPPINGS['Technology'] == energy_source]["Category"].iloc[0])
        return ret

    elif id == 11: # voluntary components of the commitment associated with the farthest date
        voluntary_perc = float(row.iloc[:, 17].values[0])
        if voluntary_perc <= 0:
            return "No"
        else:
            return 'Yes'

    else:
        return "QUESTION ID NOT RECOGNIZED"

In [21]:
queries = []

count = 0
for index, row in QUESTIONS_DF.iterrows():
    query_dict = {}
    query_dict['question'] = row[0]
    query_dict['type'] = row[1]
    query_dict['tags'] = row[2]
    query_dict['id'] = int(row[3])
    queries.append(query_dict)

files_to_query = []
with open(f"{DATA_DIR}/{FILES_TO_QUERY_TXT}", "r") as file:
    files_to_query = [line.strip() for line in file]

/scratch/local/52577217/ipykernel_1145633/3989365342.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  query_dict['question'] = row[0]
/scratch/local/52577217/ipykernel_1145633/3989365342.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  query_dict['type'] = row[1]
/scratch/local/52577217/ipykernel_1145633/3989365342.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  query_dict['tags'] = row[2]
/scratch/local/52577217/ipykernel

In [24]:
# load llm
HF_LLM_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3" # hardcoded mistral
llm = HuggingFaceEndpoint(
    repo_id=HF_LLM_MODEL_ID,
    huggingfacehub_api_token=HF_API_KEY,
)
model = "Mistral"

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /blue/rcstudents/smaley/alp/ALP_updated/models/token
Login successful


In [25]:
# build retriever
from langchain.retrievers.multi_query import MultiQueryRetriever
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(), llm=llm
)

# build prompt
prompt = hub.pull("rlm/rag-prompt")
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
from langchain_core.runnables import RunnableParallel
rag_chain_from_docs = (
    RunnablePassthrough.assign(context=(lambda x: format_docs(x["context"])))
    | prompt
    | llm
    | StrOutputParser()
)

# intialize rag_chain
rag_chain_with_source = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()} # change retriever as necessary
).assign(answer=rag_chain_from_docs)

df = pd.DataFrame(columns=["Type","Tags","Question ID","Question","Ground Truth","Response","Source"])
# iterate through questions
for query in queries:
    prompt = f"{query['question']}\nAddtional Information: {query['tags']}"
    answer = rag_chain_with_source.invoke(prompt)
    # print(answer)
    ground_truth = retrieve_ground_truth_values(GT_DF, query['id'], reference_law) # hard-coded ground-truth values
    row = [query['type'], query['tags'], query['id'], query['question'], ground_truth, answer['answer'], answer['context']]
    df.loc[len(df)] = row
filename = OUTPUT_DIR + "/" + reference_law + "_(" + model + ").xlsx"
with pd.ExcelWriter(filename) as writer:
    df.to_excel(writer)

/blue/daisyw/smaley/.conda/envs/alp-env/lib/python3.12/site-packages/langsmith/client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [29]:
OUTPUT_DIR = "../data/embeddings"
for subdirs, dirs, _ in os.walk(OUTPUT_DIR):
    for dir_name in dirs:
        print(dir_name)

MD SB 516
MD SB 791
reference_law
Guam Public Statutes 29-62
